# 06 — Generate Paraphrased AI Test Set


## 1. Environment


In [1]:
import sys
import os
import re
import json
import hashlib
import importlib.metadata
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/spotbot/prepared.py").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))
from spotbot.prepared import load_prepared_splits

folder = REPO_ROOT / "data/processed/robustness"
folder.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HOME", str(REPO_ROOT / "models/huggingface-home"))
DOMAINS = ["finance", "medicine", "open_qa", "reddit_eli5", "wiki_csai"]
KEY = ["question_group_id", "label", "answer_index"]
MODEL = "humarin/chatgpt_paraphraser_on_T5_base"
REVISION = "d3ab1136f6c3d3d2f8031d07551d2054a5372a50"
REGENERATE = not all((folder / name).exists() for name in ["paraphrased_ai_pilot.csv", "manifest.json"])  # Set True to regenerate.
print("Repository:", REPO_ROOT)
print("Python:", sys.version.split()[0])
display(pd.Series({p: importlib.metadata.version(p) for p in
    ["numpy", "pandas", "torch", "transformers", "sentencepiece", "huggingface-hub"]}, name="version"))


Repository: /Users/Melinda/Desktop/Chewy-1A-spot-the-bot-detecting-ai-generated-text
Python: 3.11.13


numpy              1.26.4
pandas              2.3.3
torch               2.2.2
transformers       4.39.3
sentencepiece       0.2.3
huggingface-hub    0.36.0
Name: version, dtype: object

## 2. Load the frozen split


In [2]:
train_df, val_df, test_df = load_prepared_splits()
display(pd.DataFrame({"split": ["train", "validation", "test"],
                      "rows": [len(train_df), len(val_df), len(test_df)]}))
ai_test_df = test_df.loc[test_df.label.eq(1) & test_df.split.eq("test")].copy()
print("Available AI test examples:", len(ai_test_df))
display(ai_test_df.source.value_counts().rename("available_ai_answers"))


,split,rows
0,train,65401
1,validation,8218
2,test,8151


Available AI test examples: 2615


source
reddit_eli5    1594
finance         449
open_qa         355
medicine        134
wiki_csai        83
Name: available_ai_answers, dtype: int64

## 3. Reproduce the balanced pilot and finance replacement


In [3]:
sample_df = pd.concat([
    ai_test_df.loc[ai_test_df.source.eq(domain)].sample(n=20, random_state=42)
    for domain in DOMAINS
], ignore_index=True)
bad_mask = sample_df.text.str.contains(
    "content may violate our content policy", case=False, na=False)
assert bad_mask.sum() == 1
assert sample_df.loc[bad_mask, "source"].tolist() == ["finance"]
print("Excluded artifact:")
display(sample_df.loc[bad_mask, ["question_group_id", "source", "text"]])
clean_sample_df = sample_df.loc[~bad_mask].copy()
replacement_pool = ai_test_df.loc[
    ai_test_df.source.eq("finance")
    & ~ai_test_df.question_group_id.isin(clean_sample_df.question_group_id)
    & ~ai_test_df.text.str.contains("content may violate our content policy", case=False, na=False)
]
replacement = replacement_pool.sample(n=1, random_state=43)
pilot = pd.concat([clean_sample_df, replacement], ignore_index=True)
print("Replacement:")
display(replacement[["question_group_id", "source", "question"]])
assert len(pilot) == 100 and pilot.label.eq(1).all()
assert pilot.source.value_counts().to_dict() == dict.fromkeys(DOMAINS, 20)
assert not pilot.duplicated(KEY).any()
display(pilot.groupby(["source", "label"]).size().rename("pilot_answers"))


Excluded artifact:


,question_group_id,source,text
2,qg_47f8371e42aa5f08,finance,!This content may violate our content policy. ...


Replacement:


,question_group_id,source,question
1014,qg_2132fdd5a11e106d,finance,"How to find a good third-party, 401k managemen..."


source       label
finance      1        20
medicine     1        20
open_qa      1        20
reddit_eli5  1        20
wiki_csai    1        20
Name: pilot_answers, dtype: int64

## 4. Define the local paraphraser


In [4]:
class LocalParaphraser:
    """CPU deterministic decoding, sentence chunks, no input truncation."""
    def __init__(self, revision, cache_dir):
        import torch
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        torch.manual_seed(42)
        torch.set_num_threads(4)
        torch.use_deterministic_algorithms(True)
        self.torch = torch
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=revision, cache_dir=cache_dir)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(
            MODEL, revision=revision, cache_dir=cache_dir, use_safetensors=True).eval()
        self.revision = self.model.config._commit_hash

    def chunks(self, text):
        # Preserve every source character (apart from whitespace); split long sentences by words.
        for sentence in re.split(r'(?<=[.!?])\s+|\n+', text.strip()):
            chunk = ''
            for word in sentence.split():
                candidate = (chunk + ' ' + word).strip()
                if len(self.tokenizer.encode('paraphrase: ' + candidate)) > 120:
                    if not chunk:
                        raise ValueError('Single word exceeds model input budget')
                    yield chunk
                    chunk = word
                else:
                    chunk = candidate
            if chunk:
                if len(self.tokenizer.encode('paraphrase: ' + chunk)) > 120:
                    raise ValueError('Chunk exceeds input budget')
                yield chunk

    def __call__(self, text):
        outputs = []
        for chunk in self.chunks(text):
            inputs = self.tokenizer('paraphrase: ' + chunk, return_tensors='pt', truncation=False)
            with self.torch.inference_mode():
                tokens = self.model.generate(**inputs, do_sample=False, num_beams=4,
                                             max_new_tokens=192, early_stopping=True)
            if tokens[0, -1].item() != self.tokenizer.eos_token_id:
                raise ValueError('Output hit token limit; refusing truncated paraphrase')
            output = self.tokenizer.decode(tokens[0], skip_special_tokens=True).strip()
            if not output:
                raise ValueError('Empty model output')
            outputs.append(output)
        return ' '.join(outputs)


## 5. Generate all answers, or reopen the saved run


In [5]:
output = folder / "paraphrased_ai_pilot.csv"
manifest_path = folder / "manifest.json"
if REGENERATE:
    engine = LocalParaphraser(REVISION, str(REPO_ROOT / "models/huggingface"))
    paraphrased_df = pilot.copy()
    paraphrased_df["original_text"] = pilot.text
    paraphrases = []
    for i, text in enumerate(pilot.text, start=1):
        paraphrases.append(engine(text))
        print(f"Generated {i}/100", flush=True)
    paraphrased_df["paraphrased_text"] = paraphrases
    paraphrased_df["review_status"] = "pending"
    paraphrased_df["length_ratio"] = (
        paraphrased_df.paraphrased_text.str.len() / paraphrased_df.original_text.str.len())
else:
    if not output.exists() or not manifest_path.exists():
        raise FileNotFoundError("Saved run missing. Set REGENERATE = True and rerun from the top.")
    paraphrased_df = pd.read_csv(output, keep_default_na=False)
    print("Loaded saved candidates:", len(paraphrased_df))


Loaded saved candidates: 100


## 6. Validate metadata and outputs


In [6]:
def validate(pilot, result):
    """Fail on missing/extra/reordered rows, changed metadata, or blank outputs."""
    if len(pilot) != 100 or pilot.source.value_counts().to_dict() != dict.fromkeys(DOMAINS, 20):
        raise ValueError('Expected 20 pilot answers per domain')
    if not pilot.label.eq(1).all() or not pilot.split.eq('test').all() or pilot.duplicated(KEY).any():
        raise ValueError('Expected unique AI test answers')
    pd.testing.assert_frame_equal(pilot.reset_index(drop=True), result[pilot.columns].reset_index(drop=True))
    if not result.original_text.equals(result.text):
        raise ValueError('Original text misaligned')
    if not result.paraphrased_text.map(lambda x: isinstance(x, str) and bool(x.strip())).all():
        raise ValueError('Empty paraphrase')
    if result.paraphrased_text.eq(result.original_text).any():
        raise ValueError('Unchanged paraphrase')

validate(pilot, paraphrased_df)
print("PASS: counts, labels, domains, identifiers, row order, original text, nonempty and changed outputs")
display(paraphrased_df.groupby(["source", "label"]).size().rename("generated_answers"))
display(paraphrased_df.length_ratio.describe().rename("paraphrase/original character ratio"))


PASS: counts, labels, domains, identifiers, row order, original text, nonempty and changed outputs


source       label
finance      1        20
medicine     1        20
open_qa      1        20
reddit_eli5  1        20
wiki_csai    1        20
Name: generated_answers, dtype: int64

count    100.000000
mean       0.897854
std        0.100737
min        0.565150
25%        0.859403
50%        0.921398
75%        0.954955
max        1.063683
Name: paraphrase/original character ratio, dtype: float64

## 7. Save the dataset and reproducibility manifest


In [7]:
if REGENERATE:
    pilot.to_csv(folder / "pilot_originals.csv", index=False)
    paraphrased_df.to_csv(output, index=False)
    manifest = {
        'model': MODEL, 'revision': engine.revision, 'device': 'cpu', 'seed': 42,
        'selection_seeds': [42, 43], 'rows': len(paraphrased_df),
        'domains': paraphrased_df.source.value_counts().to_dict(),
        'generation': {'do_sample': False, 'num_beams': 4, 'max_new_tokens': 192, 'early_stopping': True},
        'chunking': 'sentence/newline then whitespace; maximum 120 input tokens including prefix',
        'versions': {p: importlib.metadata.version(p) for p in ['numpy', 'pandas', 'torch', 'transformers', 'sentencepiece', 'huggingface-hub']},
        'frozen_split_manifest': json.loads((REPO_ROOT / 'data/splits/manifest.json').read_text()),
        'sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in [output, folder / 'pilot_originals.csv']},
        'review': 'Automated structural checks pass; semantic equivalence requires human review.',
    }
    (folder / 'manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
else:
    manifest = json.loads(manifest_path.read_text())
    assert manifest["model"] == MODEL and manifest["revision"] == REVISION
    for filename, expected in manifest["sha256"].items():
        assert hashlib.sha256((folder / filename).read_bytes()).hexdigest() == expected, filename
    pd.testing.assert_frame_equal(pilot, pd.read_csv(folder / "pilot_originals.csv", keep_default_na=False))
    print("PASS: saved originals and dataset checksums")
print("Output:", output.relative_to(REPO_ROOT))
print("Model revision:", manifest["revision"])


PASS: saved originals and dataset checksums
Output: data/processed/robustness/paraphrased_ai_pilot.csv
Model revision: d3ab1136f6c3d3d2f8031d07551d2054a5372a50


## 8. Inspect original/paraphrase pairs


In [8]:
review_rows = [int(g.index[0]) for _, g in paraphrased_df.groupby("source")]
for i in review_rows:
    row = paraphrased_df.loc[i]
    display(Markdown(f"### Row {i}: {row.source} — `{row.question_group_id}`"))
    display(pd.DataFrame({"version": ["original", "paraphrased"],
                          "text": [row.original_text, row.paraphrased_text]}).style.set_properties(
                              **{"white-space": "pre-wrap", "text-align": "left"}))


### Row 0: finance — `qg_9c699ccd4377140a`

Matplotlib is building the font cache; this may take a moment.


,version,text
0,original,"Equity refers to the ownership interest in a company. It represents the residual value that would be left over for shareholders if all of a company's debts were paid off. In other words, equity is the value of the company that is owned by the shareholders. This can be represented in the form of stock or shares in the company. Equity can also refer to the difference between the value of an asset, such as a home, and any debts or liabilities associated with that asset. For example, if you own a home worth $500,000 and you have a mortgage balance of $400,000, your equity in the home would be $100,000."
1,paraphrased,"Equity is the percentage of ownership in a company. If a company's debts were fully paid off, the residual value that would remain for shareholders is what it represents. The value of the company owned by its shareholders is referred to as equity. This can be conveyed through the use of stock or shares in the company. The value of an asset, like a house, can be divided by its debts or liabilities, which is known as equity. If you possess a $500,000 home and have a $400,000 mortgage balance, your home's equity would be $100,000."


### Row 19: medicine — `qg_ec5ed95a54f06577`

,version,text
0,original,"It is not possible for me to diagnose the cause of a lump or any other medical issue without examining the individual and obtaining a complete medical history. It is important to see a healthcare provider for an evaluation and proper diagnosis. There are many potential causes for a lump above the knee, including a cyst, a lipoma (a benign fatty tumor), a hematoma (a collection of blood outside of a blood vessel), or a muscle strain or injury. It is important to see a healthcare provider for an evaluation to determine the cause and appropriate treatment. In the meantime, you should avoid activities that cause discomfort or pain, and try to keep the affected area elevated and supported to help reduce swelling."
1,paraphrased,"Without examining the individual and gathering a complete medical history, I cannot determine the cause of a lump or any other medical condition. Consultation with a physician is necessary for evaluation and accurate diagnosis. A lump above the knee can be caused by a cyst, lipoma, hematoma, or muscle strain or injury. Consultation with a physician is necessary to determine the cause and treatment options. During this time, refrain from engaging in activities that cause discomfort and instead maintain a level and supportive surface to alleviate the swelling."


### Row 39: open_qa — `qg_a2e445c20c774f9d`

,version,text
0,original,"There are seven professional hockey teams in Canada:\n\nCalgary Flames (National Hockey League)\nEdmonton Oilers (National Hockey League)\nMontreal Canadiens (National Hockey League)\nOttawa Senators (National Hockey League)\nToronto Maple Leafs (National Hockey League)\nVancouver Canucks (National Hockey League)\nWinnipeg Jets (National Hockey League)\nAll of these teams are members of the National Hockey League (NHL), which is the premier professional ice hockey league in the world. The NHL is composed of 31 teams, with 24 located in the United States and seven located in Canada."
1,paraphrased,"In Canada, there are seven professional hockey teams: the Calgary Flames, Edmonton Oilers, Montreal Canadiens, Ottawa Senators, Vancouver Maple Leafs, Vancouver Canucks, and Winnipeg Jets. The National Hockey League (NHL) is the top professional ice hockey league globally. There are 31 NHL teams, with 24 in the United States and seven in Canada."


### Row 59: reddit_eli5 — `qg_958e5c48c2e763af`

,version,text
0,original,"Great question! Gas stations list the price of fuel in terms of how many tenths of a gallon you are buying, rather than the price of a full gallon, because it allows them to charge for the fuel more accurately. Imagine that you want to buy a gallon of fuel for your car. The gas pump has a meter that measures how much fuel you are getting, but it might not be able to measure exactly how much fuel you are getting down to the last drop. The meter might stop at 0.9 gallons, for example, even though you were actually getting a little bit more than that. If the gas station charged you the full price for a gallon of fuel in this situation, you would be paying more than you should. By listing the price in terms of how many tenths of a gallon you are buying, the gas station can charge you more accurately for the fuel you are actually getting. So, if you see that the price of fuel is listed as $3.50 per 9/10 of a gallon, that means that you are paying $3.50 for every tenth of a gallon of fuel that you get. If the gas pump measures that you are getting 0.9 gallons of fuel, then you will pay $3.50 x 9 = $31.50 for that fuel. I hope this helps to clarify things! Let me know if you have any other questions."
1,paraphrased,"That's a great inquiry. The price of fuel is listed by gas stations in the tenths of a gallon instead of the full gallon, which enables them to charge for the fuel more accurately. Pretend that you are willing to spend a gallon of gas on your automobile. The gas pump has a meter that measures the quantity of fuel you are consuming, but it may not be able to accurately gauge the quantity of fuel you are consuming to the lowest level. The meter could come to a halt at 0.9 gallons, even though you were receiving a bit more than what was being measured. In this scenario, if you were charged the full price for a gallon of fuel at the gas station, you would be paying more than you should. The gas station can accurately charge you for the fuel you are getting by listing the price in terms of the tenths of a gallon. If the price of fuel is $3.50 per 9/10 of a gallon, then you are paying $3.50 for every tenth of a gallon of fuel. $31.50 is the cost of 0.9 gallons of fuel at the gas pump, which is determined by $3.50 x 9 = $31.50. I hope this elucidates the situation! If you have additional questions, please let me know."


### Row 79: wiki_csai — `qg_4bf5c0334d9cce63`

,version,text
0,original,"Bounded rationality is a concept in behavioral economics and decision-making that suggests that when individuals make decisions, their rationality is limited by the cognitive and emotional constraints of their minds. This means that people's decision-making is not always fully rational or logical, but rather is influenced by their biases, emotions, and other mental limitations. The concept of bounded rationality was developed by economist Herbert Simon in the 1950s and suggests that people's decision-making is not always fully consistent with the ideal of rationality as defined in traditional economics. Instead, people's decision-making is shaped by their limited cognitive abilities and the information that is available to them. Bounded rationality is an important concept in understanding how people make decisions in real-world situations, as it acknowledges that people's decision-making is not always fully rational or based on complete information. It is also an important consideration in fields such as management and public policy, where decision-making is often based on incomplete or uncertain information."
1,paraphrased,"The concept of bounded rationality in behavioral economics and decision-making posits that individuals' rationality is constrained by the cognitive and emotional constraints of their minds when making decisions. People's decision-making is often influenced by their biases, emotions, and mental limitations, rather than a logical one. Herbert Simon, an economist, introduced the concept of bounded rationality in the 1950s, which suggests that people's decision-making may not always align with the concept of rationality as defined in conventional economics. Individuals' decision-making is influenced by their limited intelligence and the information they have at their disposal. The concept of bounded rationality is significant in comprehending people's decision-making in real-life scenarios, as it recognizes that decision-making is not always entirely rational or based on complete information. The importance of this factor cannot be overstated in fields like management and public policy, where decision-making is often based on incomplete or uncertain information."


## 9. Known semantic failures and review queue


In [9]:
findings = pd.DataFrame([
    (0, "Equity definition changes a difference into division."),
    (19, "Elevating the affected area becomes a level surface."),
    (39, "Toronto Maple Leafs becomes Vancouver Maple Leafs."),
    (53, "Mathematical conditions and formulas are omitted.")
], columns=["pilot_row", "review_finding"])
findings = findings.join(paraphrased_df[["question_group_id", "source", "answer_index"]], on="pilot_row")
display(findings)
print("Candidates requiring semantic review:", paraphrased_df.review_status.eq("pending").sum())
display(paraphrased_df.nsmallest(10, "length_ratio")[[
    "question_group_id", "source", "length_ratio", "review_status"]])


,pilot_row,review_finding,question_group_id,source,answer_index
0,0,Equity definition changes a difference into di...,qg_9c699ccd4377140a,finance,0
1,19,Elevating the affected area becomes a level su...,qg_ec5ed95a54f06577,medicine,1
2,39,Toronto Maple Leafs becomes Vancouver Maple Le...,qg_a2e445c20c774f9d,open_qa,1
3,53,Mathematical conditions and formulas are omitted.,qg_5bbcd1e1696d51a5,open_qa,1


Candidates requiring semantic review: 100


,question_group_id,source,length_ratio,review_status
53,qg_5bbcd1e1696d51a5,open_qa,0.565150,pending
39,qg_a2e445c20c774f9d,open_qa,0.590136,pending
3,qg_29047ce44b70102b,finance,0.632143,pending
14,qg_e4f00f3d3988bad6,finance,0.634527,pending
18,qg_2dcd14701a305374,finance,0.643519,pending
65,qg_c0952244c2f7e0da,reddit_eli5,0.686016,pending
2,qg_c099b344813426bc,finance,0.695051,pending
40,qg_24ada53d9f6d1b92,open_qa,0.733380,pending
16,qg_86a0a06799e37733,finance,0.771408,pending
55,qg_db4df4717d857f78,open_qa,0.771670,pending
